# Pharmaceutical Sales Prediction — Linear Regression

Build a regression model using Python and scikit-learn to predict `monthly_sales_usd`.

**Dataset:** `pharma_sales_regression_dataset.csv`

> Synthetic educational data; not real Novartis or confidential commercial data.


## 1. Business Problem

Predict monthly pharmaceutical sales using marketing spend, doctor visits, samples,
previous-month sales, competition, stock availability, discounts, market potential,
sales representatives, product, region, channel and quarter.

**Target:** `monthly_sales_usd`

This is a **regression** problem because the target is continuous numerical data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")


In [ ]:
# Load data
df = pd.read_csv("pharma_sales_regression_dataset.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()


In [ ]:
# Inspect dataset
df.info()


In [ ]:
# Summary statistics
df.describe(include="all").T


In [ ]:
# Missing-value check
df.isnull().sum()


## 2. Exploratory Data Analysis

In [ ]:
# Sales distribution
plt.figure(figsize=(9,5))
plt.hist(df["monthly_sales_usd"], bins=30)
plt.xlabel("Monthly Sales (USD)")
plt.ylabel("Frequency")
plt.title("Distribution of Monthly Pharmaceutical Sales")
plt.show()


In [ ]:
# Marketing spend vs sales
plt.figure(figsize=(9,5))
plt.scatter(df["marketing_spend_usd"], df["monthly_sales_usd"], alpha=0.6)
plt.xlabel("Marketing Spend (USD)")
plt.ylabel("Monthly Sales (USD)")
plt.title("Marketing Spend vs Monthly Sales")
plt.show()


In [ ]:
# Doctor visits vs sales
plt.figure(figsize=(9,5))
plt.scatter(df["doctor_visits"], df["monthly_sales_usd"], alpha=0.6)
plt.xlabel("Doctor Visits")
plt.ylabel("Monthly Sales (USD)")
plt.title("Doctor Visits vs Monthly Sales")
plt.show()


In [ ]:
# Average sales by product
(df.groupby("product")["monthly_sales_usd"]
   .mean()
   .sort_values(ascending=False)
   .plot(kind="bar", figsize=(9,5)))

plt.ylabel("Average Monthly Sales (USD)")
plt.title("Average Sales by Product")
plt.xticks(rotation=45)
plt.show()


## 3. Features and Target

In [ ]:
target = "monthly_sales_usd"

X = df.drop(columns=[target, "record_id"])
y = df[target]

categorical_features = [
    "region", "product", "sales_channel", "quarter"
]

numerical_features = [
    "sales_rep_count",
    "marketing_spend_usd",
    "doctor_visits",
    "samples_distributed",
    "prior_month_sales_usd",
    "competitor_index",
    "stock_availability",
    "discount_pct",
    "market_potential_index"
]

print("Target:", target)
print("Categorical:", categorical_features)
print("Numerical:", numerical_features)


## 4. Train/Test Split

Use 80% of observations for training and 20% for testing.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 5. Preprocessing + Linear Regression

Categorical variables are converted using `OneHotEncoder`.
The preprocessing and regression model are combined into one pipeline.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical",
         OneHotEncoder(handle_unknown="ignore"),
         categorical_features),
        ("numerical",
         "passthrough",
         numerical_features)
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)


In [ ]:
# Train model
model.fit(X_train, y_train)

print("Model trained successfully.")


## 6. Predictions

In [ ]:
y_pred = model.predict(X_test)

prediction_df = pd.DataFrame({
    "Actual_Sales": y_test.values,
    "Predicted_Sales": y_pred,
    "Residual": y_test.values - y_pred
})

prediction_df.head(10)


## 7. Model Evaluation

- **MAE:** average absolute error
- **MSE:** average squared error
- **RMSE:** error in the same unit as sales
- **R²:** proportion of target variance explained by the model


In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE  : ${mae:,.2f}")
print(f"MSE  : {mse:,.2f}")
print(f"RMSE : ${rmse:,.2f}")
print(f"R²   : {r2:.4f}")


In [ ]:
# Actual vs predicted
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred, alpha=0.7)

lo = min(y_test.min(), y_pred.min())
hi = max(y_test.max(), y_pred.max())

plt.plot([lo, hi], [lo, hi], linestyle="--")
plt.xlabel("Actual Sales (USD)")
plt.ylabel("Predicted Sales (USD)")
plt.title("Actual vs Predicted Pharmaceutical Sales")
plt.show()


In [ ]:
# Residual plot
residuals = y_test - y_pred

plt.figure(figsize=(9,5))
plt.scatter(y_pred, residuals, alpha=0.7)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Sales (USD)")
plt.ylabel("Residual")
plt.title("Residual Plot")
plt.show()


## 8. Interpret Model Coefficients

In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["regressor"].coef_

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

coef_df["Absolute_Coefficient"] = coef_df["Coefficient"].abs()

coef_df.sort_values(
    "Absolute_Coefficient",
    ascending=False
).head(20)


A positive coefficient indicates a positive modeled association with predicted sales,
while a negative coefficient indicates a negative modeled association.

**Training note:** coefficients show association in this model; they do not prove causation.


## 9. New Business Scenario Prediction

In [ ]:
new_case = pd.DataFrame({
    "region": ["West"],
    "product": ["OncoPlus"],
    "sales_channel": ["Hospital"],
    "quarter": ["Q3"],
    "sales_rep_count": [55],
    "marketing_spend_usd": [120000],
    "doctor_visits": [350],
    "samples_distributed": [9000],
    "prior_month_sales_usd": [450000],
    "competitor_index": [0.85],
    "stock_availability": [0.95],
    "discount_pct": [8],
    "market_potential_index": [1.20]
})

predicted_sales = model.predict(new_case)[0]
print(f"Predicted Monthly Sales: ${predicted_sales:,.2f}")


In [ ]:
# Compare two commercial scenarios
scenario_1 = new_case.copy()

scenario_2 = new_case.copy()
scenario_2["marketing_spend_usd"] = 160000
scenario_2["doctor_visits"] = 420

pred_1 = model.predict(scenario_1)[0]
pred_2 = model.predict(scenario_2)[0]

print(f"Scenario 1: ${pred_1:,.2f}")
print(f"Scenario 2: ${pred_2:,.2f}")
print(f"Difference: ${(pred_2 - pred_1):,.2f}")


## 10. Student Challenges

1. Remove `prior_month_sales_usd` and compare R².
2. Train a model using only numerical features.
3. Identify the three strongest coefficients.
4. Change `competitor_index` and observe predicted sales.
5. Change `stock_availability` and observe predicted sales.
6. Compare Linear Regression with Ridge Regression.
7. Compare Linear Regression with Lasso Regression.
8. Calculate percentage prediction error.
9. Explain the model to a pharmaceutical commercial manager in business language.


## Key Takeaways

**Business problem → Data → Features → Encoding → Train/Test Split → Linear Regression → Prediction → Evaluation → Business Interpretation**

The same workflow can later be extended to more advanced forecasting models.
